# 03 — Byte-Level BPE

In the previous notebook, we implemented the core BPE idea using characters.

Now let's move one level closer to a real LLM tokenizer.

Instead of starting with characters:

text → characters → BPE

we will start with bytes:

text → UTF-8 bytes → BPE

## Learning Goals

- Understand UTF-8 bytes
- Understand why byte-level BPE starts with 256 possible byte values
- Convert text into bytes
- Build the initial byte vocabulary
- Run BPE merges on byte sequences
- Store the vocabulary and merge rules
- Build `encode()` and `decode()`
- Train the tokenizer on our TinyStories corpus
- Measure the resulting token count and compression

## Main Pipeline
```
Text
↓
UTF-8 encoding
↓
Bytes
↓
Initial byte vocabulary
↓
Pair counting
↓
BPE merges
↓
Final vocabulary
↓
Encoding
↓
Token IDs
```

In [ ]:
# Step 1 — See what UTF-8 gives
text = "hello"

byte_data = text.encode("utf-8")

print(byte_data)
print(list(byte_data))

b'hello'
[104, 101, 108, 108, 111]


In [ ]:
# Step 2 — Use a non-ASCII example
text = "café"

byte_data = text.encode("utf-8")

print(byte_data)
print(list(byte_data))

b'caf\xc3\xa9'
[99, 97, 102, 195, 169]


## Observation

UTF-8 represents text as bytes.

ASCII characters generally occupy one byte, while some Unicode characters
require multiple bytes.

A byte can always be represented using a value from 0 to 255.

Therefore, a byte-level tokenizer can begin with a fixed base vocabulary of
256 possible byte values instead of learning a character vocabulary from the
training corpus.

In [3]:
# Step 3 — Create the initial byte vocabulary

byte_vocab = list(range(256))

print("Vocabulary size:", len(byte_vocab))
print("First 10:", byte_vocab[:10])
print("Last 10:", byte_vocab[-10:])

Vocabulary size: 256
First 10: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
Last 10: [246, 247, 248, 249, 250, 251, 252, 253, 254, 255]


In [13]:
# Step 4 — Convert text to bytes and treat them as initial tokens
text = "play"

byte_sequence = list(text.encode("utf-8"))

print("Text:", text)
print("Bytes:", byte_sequence)

tokens = byte_sequence

print("Initial tokens:", tokens)

Text: play
Bytes: [112, 108, 97, 121]
Initial tokens: [112, 108, 97, 121]


In [14]:
# Step 5 — Count byte pairs
pairs = list(zip(tokens, tokens[1:]))

print("Adjacent pairs:", pairs)

Adjacent pairs: [(112, 108), (108, 97), (97, 121)]


In [15]:
# Step 6 — Create the first learned token
merge_pair = (112, 108)
new_token_id = 256

token_bytes = {
    new_token_id: merge_pair
}

tokens = [256, 97, 121]

print("Tokens after merge:", tokens)
print("New token:", new_token_id)
print("Represents bytes:", token_bytes[new_token_id])

Tokens after merge: [256, 97, 121]
New token: 256
Represents bytes: (112, 108)


In [23]:
# Step 7 — Store token -> bytes and decode

# base byte vocab
vocab = {i: bytes([i]) for i in range(256)}

# add the first learned bpe token
vocab[256] = vocab[112] + vocab[108]

print("Token 112:", vocab[112])
print("Token 108:", vocab[108])
print("Token 256:", vocab[256])

Token 112: b'p'
Token 108: b'l'
Token 256: b'pl'


In [25]:
# Step 8 — Decode token sequence
tokens = [256, 97, 121]

byte_data = b"".join(vocab[token] for token in tokens)

print("Bytes:", byte_data)
print("Decoded:", byte_data.decode("utf-8"))

Bytes: b'play'
Decoded: play


## Observation

A learned BPE token does not replace the underlying bytes.

Token `256` represents the byte sequence `[112, 108]`, which corresponds to
the text `"pl"`.

During decoding, token IDs are mapped back to their byte sequences and joined.
The resulting bytes are then decoded as UTF-8 to reconstruct the original text.

Therefore, the tokenizer must maintain a mapping between:

token ID → byte sequence

This mapping will eventually be part of our real tokenizer vocabulary.

In [36]:
# Step 9 — Build a tiny byte-level corpus
words = [
    "play",
    "played",
    "playing",
    "player",
    "plays",
    "replay",
    "replayed",
    "replaying"
]

corpus = [list(word.encode("utf-8")) for word in words]

for word, tokens in zip(words, corpus):
    print(word, "->", tokens)

play -> [112, 108, 97, 121]
played -> [112, 108, 97, 121, 101, 100]
playing -> [112, 108, 97, 121, 105, 110, 103]
player -> [112, 108, 97, 121, 101, 114]
plays -> [112, 108, 97, 121, 115]
replay -> [114, 101, 112, 108, 97, 121]
replayed -> [114, 101, 112, 108, 97, 121, 101, 100]
replaying -> [114, 101, 112, 108, 97, 121, 105, 110, 103]


In [37]:
# Step 10 — Count byte pairs
def count_pairs(corpus):
    pair_counts = {}

    for word in corpus:
        for pair in zip(word, word[1:]):
            pair_counts[pair] = pair_counts.get(pair, 0) + 1

    return pair_counts

pair_counts = count_pairs(corpus)
pair = max(pair_counts, key=pair_counts.get)

print("Most frequent pair:", pair)
print("Count:", pair_counts[pair])

Most frequent pair: (112, 108)
Count: 8


In [38]:
# Step 11 — Automatically create the new token
vocab = {i: bytes([i]) for i in range(256)}

next_token_id = 256

pair = max(pair_counts, key=pair_counts.get)

vocab[next_token_id] = (
    vocab[pair[0]] + vocab[pair[1]]
)

print("New token:", next_token_id)
print("Represents:", vocab[next_token_id])

New token: 256
Represents: b'pl'


In [39]:
# Step 12 — Merge that token across the corpus
def merge_pair(tokens, pair, new_token_id):
    merged = []
    i = 0

    while i < len(tokens):
        if (
            i < len(tokens) - 1
            and (tokens[i], tokens[i + 1]) == pair
        ):
            merged.append(new_token_id)
            i += 2
        else:
            merged.append(tokens[i])
            i += 1

    return merged

print("Before:", corpus[0])
corpus = [merge_pair(tokens, pair, 257) for tokens in corpus]

for word, tokens in zip(words, corpus):
    print(word, "->", tokens)


Before: [112, 108, 97, 121]
play -> [257, 97, 121]
played -> [257, 97, 121, 101, 100]
playing -> [257, 97, 121, 105, 110, 103]
player -> [257, 97, 121, 101, 114]
plays -> [257, 97, 121, 115]
replay -> [114, 101, 257, 97, 121]
replayed -> [114, 101, 257, 97, 121, 101, 100]
replaying -> [114, 101, 257, 97, 121, 105, 110, 103]


In [41]:
# put in a single trainign loop - which counts pairs, selects teh most frequent,
# creates a new token, stores the merge rule, merges the corpus, and increments the vocabulary

words = [
    "play",
    "played",
    "playing",
    "player",
    "plays",
    "replay",
    "replayed",
    "replaying"
]
vocab = {i: bytes([i]) for i in range(256)}
corpus = [list(word.encode("utf-8")) for word in words]
merges = {}
merge_ranks = {}

for i in range(10):
    pair_counts = count_pairs(corpus)
    pair = max(pair_counts, key=pair_counts.get)

    new_token_id = 256 + i
    merges[pair] = new_token_id
    merge_ranks[pair] = i

    corpus = [merge_pair(tokens, pair, new_token_id) for tokens in corpus]
    vocab[new_token_id] = vocab[pair[0]] + vocab[pair[1]]

    print(
        f"Step {i+1}: {pair} → {new_token_id} "
        f"({vocab[new_token_id]})"
    )
print()
print(len(vocab))
print(merges)

Step 1: (112, 108) → 256 (b'pl')
Step 2: (256, 97) → 257 (b'pla')
Step 3: (257, 121) → 258 (b'play')
Step 4: (258, 101) → 259 (b'playe')
Step 5: (114, 101) → 260 (b're')
Step 6: (259, 100) → 261 (b'played')
Step 7: (258, 105) → 262 (b'playi')
Step 8: (262, 110) → 263 (b'playin')
Step 9: (263, 103) → 264 (b'playing')
Step 10: (259, 114) → 265 (b'player')

266
{(112, 108): 256, (256, 97): 257, (257, 121): 258, (258, 101): 259, (114, 101): 260, (259, 100): 261, (258, 105): 262, (262, 110): 263, (263, 103): 264, (259, 114): 265}


## Important Observation

BPE does not replace an older token when a larger token is learned.

Each merge adds a new token to the vocabulary.

For example:

pl → pla → play → playe → played

All of these can remain vocabulary entries.

During encoding, the learned merge rules determine how far the merging process
can proceed for a particular piece of text.

In [42]:
token_id = 259

print("Token ID:", token_id)
print("Bytes:", vocab[token_id])
print("Text:", vocab[token_id].decode("utf-8"))

Token ID: 259
Bytes: b'playe'
Text: playe


In [45]:
tokens = [259, 100]

byte_data = b''.join([vocab[token] for token in tokens])
byte_data.decode('utf-8')

'played'

In [46]:
merges

{(112, 108): 256,
 (256, 97): 257,
 (257, 121): 258,
 (258, 101): 259,
 (114, 101): 260,
 (259, 100): 261,
 (258, 105): 262,
 (262, 110): 263,
 (263, 103): 264,
 (259, 114): 265}

In [50]:
merge_ranks

{(112, 108): 0,
 (256, 97): 1,
 (257, 121): 2,
 (258, 101): 3,
 (114, 101): 4,
 (259, 100): 5,
 (258, 105): 6,
 (262, 110): 7,
 (263, 103): 8,
 (259, 114): 9}

In [55]:
# let's implement the actual encode() logic using the merge ranks.

def encode(text):
    tokens = list(text.encode("utf-8"))
    # print('o:', tokens)

    while True:
        pair_candidates = []

        for i in range(len(tokens) - 1):
            pair = (tokens[i], tokens[i + 1])
            # print('1:', pair)

            if pair in merge_ranks:
                pair_candidates.append((merge_ranks[pair], pair))
                # print('2:', pair_candidates)

        if not pair_candidates:
            break

        _, best_pair = min(pair_candidates)
        # print('3:', _, best_pair)

        new_token_id = merges[best_pair]
        # print('4:', new_token_id)

        tokens = merge_pair(tokens, best_pair, new_token_id)
        # print('5:', tokens)

    return tokens

for text in ["play"]:
    tokens = encode(text)
    print(text, "->", tokens)

play -> [258]


In [57]:
def decode(tokens):
    byte_data = b"".join(vocab[token] for token in tokens)
    return byte_data.decode("utf-8")

for text in ["play", "playe", "played", "playing", "player", "playful"]:
    tokens = encode(text)
    decoded = decode(tokens)

    print(f"{text} → {tokens} → {decoded}")

play → [258] → play
playe → [259] → playe
played → [261] → played
playing → [264] → playing
player → [265] → player
playful → [258, 102, 117, 108] → playful


In [109]:
texts = ["play", "playe", "played", "playing", "player", "playful"]

for text in texts:
    encoded = encode(text)
    decoded = decode(encoded)

    assert decoded == text

print("Encode -> Decode round trip successful")

Encode -> Decode round trip successful


## Building the Full Tokenizer

The toy implementation used separate variables for:

- `vocab`
- `merges`
- `merge_ranks`
- `encode()`
- `decode()`

A real tokenizer should keep these pieces together.

Our tokenizer will therefore maintain:

```text
Tokenizer
├── vocab
├── merges
├── merge_ranks
├── special_tokens
├── train()
├── encode()
├── decode()
├── save()
└── load()


---

# Step 1 — Design the tokenizer state

We'll first create the class without implementing all methods yet.

```python
class ByteLevelBPETokenizer:

    def __init__(self):
        self.vocab = {}
        self.merges = {}
        self.merge_ranks = {}
        self.special_tokens = {}
        self.next_token_id = 0

In [93]:
import json

class ByteLevelBPETokenizer:

    def __init__(self):
        self.vocab = {}
        self.merges = {}
        self.merge_ranks = {}
        self.special_tokens = {}
        self.next_token_id = 0

    def initialize_vocab(self):
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.next_token_id = 256

    def add_special_tokens(self, special_tokens):
        for token in special_tokens:
            self.special_tokens[token] = self.next_token_id
            self.next_token_id += 1

    def count_pairs(self, corpus):
        pair_counts = {}

        for tokens in corpus:
            for pair in zip(tokens, tokens[1:]):
                pair_counts[pair] = pair_counts.get(pair, 0) + 1

        return pair_counts

    def merge_pair(self, tokens, pair, new_token_id):
        merged = []
        i = 0

        while i < len(tokens):
            if (
                i < len(tokens) - 1
                and (tokens[i], tokens[i + 1]) == pair
            ):
                merged.append(new_token_id)
                i += 2
            else:
                merged.append(tokens[i])
                i += 1

        return merged

    def train(self, texts, vocab_size):
        if not self.vocab:
            self.initialize_vocab()

        max_merges = vocab_size - 256 - len(self.special_tokens)

        corpus = [list(text.encode("utf-8")) for text in texts]

        for rank in range(max_merges):
            pair_counts = self.count_pairs(corpus)

            if not pair_counts:
                break

            pair = max(pair_counts, key=pair_counts.get)

            new_token_id = self.next_token_id
            self.next_token_id += 1

            self.merges[pair] = new_token_id
            self.merge_ranks[pair] = rank

            self.vocab[new_token_id] = (self.vocab[pair[0]] + self.vocab[pair[1]])

            corpus = [self.merge_pair(tokens, pair, new_token_id) for tokens in corpus]

            print(
                f"Merge {rank + 1}/{max_merges}: "
                f"{pair} -> {new_token_id}"
            )

    def _encode_text(self, text):
        tokens = list(text.encode("utf-8"))

        while True:
            pair_candidates = []

            for i in range(len(tokens) - 1):
                pair = (tokens[i], tokens[i + 1])

                if pair in self.merge_ranks:
                    pair_candidates.append(
                        (self.merge_ranks[pair], pair)
                    )

            if not pair_candidates:
                break

            _, best_pair = min(pair_candidates)

            tokens = self.merge_pair(
                tokens,
                best_pair,
                self.merges[best_pair]
            )

        return tokens

    def encode(self, text):
        tokens = []
        remaining = text

        for special_token, token_id in self.special_tokens.items():
            parts = remaining.split(special_token)

            for i, part in enumerate(parts):
                tokens.extend(self._encode_text(part))

                if i < len(parts) - 1:
                    tokens.append(token_id)

            remaining = ""

        if remaining:
            tokens.extend(self._encode_text(remaining))

        return tokens

    def decode(self, tokens):
        reverse_special_tokens = {
            token_id: token
            for token, token_id in self.special_tokens.items()
        }

        text = ""
        byte_data = b""

        for token in tokens:
            if token in reverse_special_tokens:
                text += byte_data.decode("utf-8")
                byte_data = b""
                text += reverse_special_tokens[token]
            else:
                byte_data += self.vocab[token]

        text += byte_data.decode("utf-8")

        return text

    def save(self, path):
        data = {
            "vocab": {
                str(token_id): list(token_bytes)
                for token_id, token_bytes in self.vocab.items()
            },
            "merges": [
                {
                    "pair": list(pair),
                    "token_id": token_id
                }
                for pair, token_id in self.merges.items()
            ],
            "merge_ranks": [
                {
                    "pair": list(pair),
                    "rank": rank
                }
                for pair, rank in self.merge_ranks.items()
            ],
            "special_tokens": self.special_tokens,
            "next_token_id": self.next_token_id
        }

        with open(path, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=2)

    def load(self, path):
        with open(path, "r", encoding="utf-8") as file:
            data = json.load(file)

        self.vocab = {
            int(token_id): bytes(token_bytes)
            for token_id, token_bytes in data["vocab"].items()
        }

        self.merges = {
            tuple(item["pair"]): item["token_id"]
            for item in data["merges"]
        }

        self.merge_ranks = {
            tuple(item["pair"]): item["rank"]
            for item in data["merge_ranks"]
        }

        self.special_tokens = data["special_tokens"]
        self.next_token_id = data["next_token_id"]

toy_words = [
    "play",
    "played",
    "playing",
    "player",
    "plays",
    "replay",
    "replayed",
    "replaying"
]

tokenizer = ByteLevelBPETokenizer()
tokenizer.initialize_vocab()
tokenizer.add_special_tokens(["<|endoftext|>"])
tokenizer.train(toy_words, vocab_size=266)
tokenizer.save("toy_tokenizer.json")

# tokenizer = ByteLevelBPETokenizer()
# tokenizer.initialize_vocab()
# print("Vocabulary size:", len(tokenizer.vocab))
# print("Next token ID:", tokenizer.next_token_id)
# tokenizer.add_special_tokens(["<|endoftext|>"])
# print(tokenizer.special_tokens)
# print("Next token ID:", tokenizer.next_token_id)

Merge 1/9: (112, 108) -> 257
Merge 2/9: (257, 97) -> 258
Merge 3/9: (258, 121) -> 259
Merge 4/9: (259, 101) -> 260
Merge 5/9: (114, 101) -> 261
Merge 6/9: (260, 100) -> 262
Merge 7/9: (259, 105) -> 263
Merge 8/9: (263, 110) -> 264
Merge 9/9: (264, 103) -> 265


In [2]:
for text in ["play", "playe", "played", "playing", "player", "playful"]:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded)

    print(text, "->", encoded, "->", decoded)

play -> [258] -> play
playe -> [259] -> playe
played -> [261] -> played
playing -> [264] -> playing
player -> [259, 114] -> player
playful -> [258, 102, 117, 108] -> playful


In [3]:
for text in ["play", "playe", "played", "playing", "player", "playful"]:
    assert tokenizer.decode(tokenizer.encode(text)) == text

print("Round-trip test passed")

Round-trip test passed


In [92]:
text = "play<|endoftext|>played play<|endoftext|>played"

tokens = tokenizer.encode(text)

print(tokens)
tokenizer.decode(tokens)

[259, 256, 262, 32, 259, 256, 262]


'play<|endoftext|>played play<|endoftext|>played'

## Important Implementation Note

The first tokenizer implementation intentionally favors clarity over speed.

For every BPE merge it:

1. Counts all adjacent pairs.
2. Selects the most frequent pair.
3. Scans the corpus and applies the merge.

This is computationally expensive for a large corpus.

That is acceptable for the learning implementation because it makes the BPE
algorithm easy to understand.

Before training thousands of merges on the full TinyStories corpus, we will
measure performance and decide what optimization is necessary.

In [97]:
loaded_tokenizer = ByteLevelBPETokenizer()
loaded_tokenizer.load("toy_tokenizer.json") # load the saved tokenizer and check if it's working as intended

text = "play<|endoftext|>played play<|endoftext|>played"
tokens = loaded_tokenizer.encode(text)
decoded = loaded_tokenizer.decode(tokens)

print(tokens)
print(decoded)

assert decoded == text
print("Save/load round trip passed")

[259, 256, 262, 32, 259, 256, 262]
play<|endoftext|>played play<|endoftext|>played
Save/load round trip passed


## Save / Load Verification

The trained tokenizer was saved to disk and loaded into a new tokenizer instance.

We then verified that the loaded tokenizer can:

- Encode text containing both normal text and the EOS special token.
- Decode the resulting token IDs.
- Reconstruct the original text exactly.

This confirms that the tokenizer state required for encoding and decoding can be persisted and restored successfully.

# Summary — What We Built

In this notebook we developed a Byte Level BPE tokenizer from scratch, starting with the core algorithm and gradually turning it into a reusable tokenizer.

## Concepts Covered

- UTF-8 encoding and byte representation
- 256-byte base vocabulary
- Difference between byte values and token IDs
- BPE pair-frequency counting
- Selecting the most frequent pair
- Creating new tokens through merges
- Growing the vocabulary through learned merges
- Storing merge rules
- Storing merge ranks / priorities
- Applying merge rules during encoding
- Encoding unseen words using learned subword/byte pieces
- Decoding token IDs back into bytes and text
- Special-token handling
- EOS / `<|endoftext|>` tokens
- Tokenizer save and load
- Encode → Decode round-trip verification

## Final Tokenizer Structure

```text
Tokenizer
├── Vocabulary
├── Merge Rules
├── Merge Ranks
├── Special Tokens
├── train()
├── encode()
├── decode()
├── save()
└── load()